In [1]:
import cipo
# Coordenadas do OASI
latitude = -8.79225   # 8°47'32,1" S
longitude = -38.68853  # 38°41'18,7" O

# Parâmetros para o cálculo
ano_inicial = 2025  # Ano de início
n_anos = 5          # Intervalo de tempo em anos (incremento de 1 a 1 ano)

janelas = cipo.calcular_janelas_observacao(latitude, longitude, ano_inicial, n_anos)

print("Janelas de Observação (15 dias com Lua Nova centralizada):\n")
for inicio, nova, fim in janelas:
    print(f"Janela: {inicio} / {fim}  |  Lua Nova: {nova}")
    

[#################################] 100% de421.bsp


Janelas de Observação (15 dias com Lua Nova centralizada):

Janela: 2025-01-22 / 2025-02-05  |  Lua Nova: 2025-01-29
Janela: 2025-02-21 / 2025-03-07  |  Lua Nova: 2025-02-28
Janela: 2025-03-22 / 2025-04-05  |  Lua Nova: 2025-03-29
Janela: 2025-04-20 / 2025-05-04  |  Lua Nova: 2025-04-27
Janela: 2025-05-20 / 2025-06-03  |  Lua Nova: 2025-05-27
Janela: 2025-06-18 / 2025-07-02  |  Lua Nova: 2025-06-25
Janela: 2025-07-17 / 2025-07-31  |  Lua Nova: 2025-07-24
Janela: 2025-08-16 / 2025-08-30  |  Lua Nova: 2025-08-23
Janela: 2025-09-14 / 2025-09-28  |  Lua Nova: 2025-09-21
Janela: 2025-10-14 / 2025-10-28  |  Lua Nova: 2025-10-21
Janela: 2025-11-13 / 2025-11-27  |  Lua Nova: 2025-11-20
Janela: 2025-12-13 / 2025-12-27  |  Lua Nova: 2025-12-20
Janela: 2026-01-11 / 2026-01-25  |  Lua Nova: 2026-01-18
Janela: 2026-02-10 / 2026-02-24  |  Lua Nova: 2026-02-17
Janela: 2026-03-12 / 2026-03-26  |  Lua Nova: 2026-03-19
Janela: 2026-04-10 / 2026-04-24  |  Lua Nova: 2026-04-17
Janela: 2026-05-09 / 2026-05

In [1]:
import pandas as pd
import numpy as np

def filter_dynamic_ephemerides(dataframes_dict, start_time, end_time, altitude_min=10, time_min_minutes=30):
    """
    Filtra as efemérides dinâmicas para encontrar janelas de observação válidas.
    
    Parâmetros:
    - dataframes_dict: Dicionário retornado por process_data_pcc_2.
    - start_time, end_time: Objetos datetime ou strings ISO para o período de busca (em UTC-3/BRT).
    - altitude_min: Altitude mínima requerida.
    - time_min_minutes: Tempo mínimo de visibilidade contínua.
    """
    resultados = {}
    
    # Padronização dos limites de tempo
    start_time = pd.to_datetime(start_time)
    end_time = pd.to_datetime(end_time)

    for obj_name, df in dataframes_dict.items():
        # Evita alterar o DataFrame original no dicionário
        df_proc = df.copy()
        
        # 1. Tratamento de Tempo: Junção de Date e UT, e conversão para UTC-3 (BRT)
        # O formato esperado na string é inferido pelo pandas. Adapte o 'format' se necessário.
        df_proc['Datetime_UTC'] = pd.to_datetime(df_proc['Date'].astype(str) + ' ' + df_proc['UT'].astype(str), errors='coerce')
        df_proc = df_proc.dropna(subset=['Datetime_UTC'])
        df_proc['Datetime_BRT'] = df_proc['Datetime_UTC'] - pd.Timedelta(hours=3)

        # 2. Filtro de Período Solicitado
        mask_period = (df_proc['Datetime_BRT'] >= start_time) & (df_proc['Datetime_BRT'] <= end_time)
        df_proc = df_proc[mask_period]
        if df_proc.empty:
            continue

        # 3. Conversão de Tipos para Filtros Numéricos
        df_proc['Object Alt'] = pd.to_numeric(df_proc['Object Alt'], errors='coerce')
        df_proc['Sun Alt'] = pd.to_numeric(df_proc['Sun Alt'], errors='coerce')

        # 4. Aplicação Simultânea de Altitude e Crepúsculo (Sol < -18°)
        mask_visible = (df_proc['Object Alt'] >= altitude_min) & (df_proc['Sun Alt'] <= -18)
        df_vis = df_proc[mask_visible].copy()
        if df_vis.empty:
            continue

        # 5. Cálculo de Janelas Contínuas de Observação
        df_vis = df_vis.sort_values('Datetime_BRT')
        df_vis['Time_Diff'] = df_vis['Datetime_BRT'].diff()
        
        # Define quebra de continuidade se a diferença entre pontos for maior que 2 horas.
        # Isso agrupa pontos sequenciais gerados pelo MPC em uma mesma "janela".
        threshold = pd.Timedelta(hours=2)
        df_vis['New_Window'] = (df_vis['Time_Diff'] > threshold) | df_vis['Time_Diff'].isna()
        df_vis['Window_ID'] = df_vis['New_Window'].cumsum()

        valid_windows = []
        for window_id, group in df_vis.groupby('Window_ID'):
            # Duração exata entre a primeira e última efeméride válida do bloco
            if len(group) > 1:
                duration_min = (group['Datetime_BRT'].max() - group['Datetime_BRT'].min()).total_seconds() / 60.0
            else:
                duration_min = 0 # 1 único ponto não atende ao requisito de tempo contínuo

            if duration_min >= time_min_minutes:
                valid_windows.append(group)

        if not valid_windows:
            continue

        # 6. Reconstrução dos Dados Válidos
        df_final = pd.concat(valid_windows)
        
        # 7. Identificação do Pico de Altitude (Max_Alt) e seu Timestamp
        idx_max = df_final['Object Alt'].idxmax()
        max_alt_val = df_final.loc[idx_max, 'Object Alt']
        max_alt_time = df_final.loc[idx_max, 'Datetime_BRT']

        # Limpeza de colunas temporárias de cálculo
        df_final = df_final.drop(columns=['Time_Diff', 'New_Window', 'Window_ID', 'Datetime_UTC'])

        resultados[obj_name] = {
            'Efemérides': df_final,
            'Max_Alt': max_alt_val,
            'Max_Alt_Time': max_alt_time.strftime('%Y-%m-%d %H:%M:%S'),
            'Janelas_Validas': len(valid_windows)
        }

    return resultados